# Module 10 — Kubernetes and TensorFlow Serving: commented walkthrough

Alexey's lecture material (lessons 10.1 – 10.8) plus the 2025 workshop, with comments that explain **three things** for every step:

- `# GOAL`   what this step is for, in plain words
- `# WHY`    the engineering logic behind it (what breaks if we skip it)
- `# PYTHON` how the Python syntax works (in Dockerfiles and YAML the third tag is `# SYNTAX`)

The original, uncommented material is untouched in `../notebooks/` (lecture notebook, `code/`, `workshop/`).

**Lecture vs homework: read this first.** The lecture (2021) deploys an image classifier as **two** services: TensorFlow Serving (C++, gRPC) plus a Flask gateway, wired with docker-compose and then with two Kubernetes deployments. **HW10 deploys none of that.** It takes the HW5 lead-scoring **FastAPI** container, loads it into a named `kind` cluster, and applies **one** Deployment (with a readiness probe), **one** `ClusterIP` Service reached through `kubectl port-forward`, and an `autoscaling/v2` HPA. The upstream material closest to that is the **workshop** (`../notebooks/workshop/`): FastAPI service, probes, requests/limits, HPA, metrics-server.

**How this notebook is built, and why.** The headings follow the lessons so you can read them side by side. The TF Serving parts (10.2 – 10.4, half of 10.7) are walked through as commented files in `commented/`, printed with `show()`: they cannot run on your Mac without the clothing SavedModel and a 2021 Python 3.8 stack, and HW10 does not need them. The runnable cells go where HW10's skills are: reading manifests, checking that labels, selectors and ports line up, and comparing the lecture's ping deployment with the workshop's production-style one (extra section after 10.7). Terminal commands are in `bash` blocks: run them in a terminal, not here (`kubectl port-forward` blocks until Ctrl-C). The commented files are for reading; build images from the complete folders in `../notebooks/`.

**What was executed while building this folder, and what was not.** Executed: this notebook top to bottom (it needs only `pyyaml`, which Jupyter itself depends on, `pandas` and `numpy`); the ping app under gunicorn with the lecture's `Pipfile.lock` on Python 3.8 (answers `PONG`); the workshop FastAPI service with the downloaded ONNX model (`/health` and `/predict` answer); every YAML parsed; every `.py` compiled. **Not executed:** Docker builds, docker-compose, TF Serving, `kind`, `kubectl` and EKS. There was no container registry access in the build environment, so no image or cluster could be started. The commands are from the lessons and the workshop README.

Differences from the lecture:
- `tf-serving-connect.ipynb` is shown as a commented script (`commented/code/tf_serving_connect.py`) instead of runnable cells: it needs TF Serving on `localhost:8500` with the SavedModel, and pins `grpcio==1.42.0` / `tensorflow-serving-api==2.7.0` from 2021, which predate your Python 3.11. `%autosave 0` and the `!pip install` cells are dropped;
- `proto.py`: `data.tostring()` → `data.tobytes()`. `ndarray.tostring` was removed in NumPy 2.3 (checked below), so the lecture line raises `AttributeError`;
- the `kube-config/*.yaml` in the repo are the **EKS** versions (10.8): their `image:` points to Alexey's ECR registry. For the local kind run (10.7) the lesson uses the local tags; the commented copies say which;
- workshop `service/uv.lock` is truncated: `uv sync --locked` (and the workshop Dockerfile) stops with *"Failed to parse uv.lock"*, and the hatchling build-system makes uv try to build the folder as a package. The commented Dockerfile uses `uv lock` once + `uv sync --locked --no-install-project` (verified);
- the workshop's `top_probability` is a **logit**, not a probability (6.4 for a pants photo in the check run, printed as 639.59%). Code unchanged, comment added; the softmax cell in 10.2 shows the conversion;
- `docker-compose` (v1) is `docker compose` in Docker Desktop, and Compose v2 warns that `version: "3.9"` is obsolete (harmless);
- MetalLB (end of 10.6) is not repeated: on macOS the kind network is not reachable from the Mac, so a `LoadBalancer` IP would not help. Port-forward is the local way in, as in HW10.

## Setup

In [ ]:
# PYTHON: pathlib.Path joins paths with '/' and works on macOS, Linux and Windows alike.
from pathlib import Path
import math

import numpy as np
import pandas as pd
import yaml          # PyYAML: parses Kubernetes manifests into Python dicts and lists

# GOAL: paths relative to this notebook (practice/).
COMMENTED = Path('commented')                 # the commented copies (read these)
LECTURE = Path('../notebooks/code')           # Alexey's originals, complete folders (build from these)
WORKSHOP = Path('../notebooks/workshop')


def show(path):
    """Print a text file, so a commented script or manifest can be read inside the notebook."""
    # PYTHON: a docstring (the triple-quoted line) documents the function; help(show) prints it.
    print(Path(path).read_text())


def load(path):
    """Parse a YAML manifest. Returns one dict, or a list of dicts if the file has several '---' documents."""
    # WHY: safe_load_all never executes code from the file (plain yaml.load can); kubectl accepts several
    #      objects in one file separated by '---', so read them all.
    docs = [d for d in yaml.safe_load_all(Path(path).read_text()) if d]
    return docs[0] if len(docs) == 1 else docs

## 10.1 Overview

The lecture's system has **two tiers**: a light **gateway** (download the image, resize, convert to protobuf; CPU work) and the **model server** (TF Serving; matrix multiplications, GPU-friendly). Separate tiers can be **scaled separately**: five cheap gateway pods in front of two expensive model pods.

| | Lecture (10.2 – 10.8) | Workshop (2025) | HW10 (2026) |
|---|---|---|---|
| Model | Keras Xception, TF SavedModel | PyTorch MobileNet, ONNX | HW5 logistic regression, `pipeline.bin` |
| Serving | TF Serving (gRPC) + Flask gateway | one FastAPI app with ONNX Runtime | one FastAPI app (`predict.py`) |
| Images | 2 (model, gateway) | 1 | 1, built from `homework/05-deployment/` |
| Local run | docker-compose | docker run | docker run |
| Kubernetes | 2 deployments + 2 services (`LoadBalancer`) | deployment + `NodePort` service + HPA | deployment + `ClusterIP` service + HPA |
| Probes, requests | no | yes | yes |
| Cluster | `kind` default name, then EKS | `kind --name mlzoomcamp` | `kind --name mlzoomcamp-2026` |

## 10.2 TensorFlow Serving

TF Serving is a C++ server that only does inference. It loads a **SavedModel** folder (not the Keras `.h5` and not TF-Lite) and answers over gRPC on port 8500. The SavedModel is converted from the Module 8 Keras model (`tf.saved_model.save(model, 'clothing-model')`); it is not in this folder, so the commands below are for reading.

```bash
# GOAL:   read the model's "signature": input/output names, dtypes and shapes. The client code needs them.
# SYNTAX: saved_model_cli ships with TensorFlow; --all prints every signature.
saved_model_cli show --dir clothing-model --all

# GOAL:   serve the model locally.
# WHY:    -v must be an ABSOLUTE path ($(pwd)), and the target needs a version folder (/1),
#         or TF Serving finds no servable. MODEL_NAME must equal the folder name under /models.
# SYNTAX: -p HOST:CONTAINER publishes a port; -v HOST:CONTAINER mounts a folder; -e sets an env variable.
docker run -it --rm \
  -p 8500:8500 \
  -v $(pwd)/clothing-model:/models/clothing-model/1 \
  -e MODEL_NAME="clothing-model" \
  tensorflow/serving:2.7.0
```

The lecture notebook as a commented script. Read it top to bottom: connect → preprocess → protobuf → request → response.

In [ ]:
show(COMMENTED / 'code/tf_serving_connect.py')

In [ ]:
# GOAL: see why the lecture says "logits, not probabilities".
# These are the 10 scores lesson 10.2 prints for the pants photo (copied from the lesson, rounded; we cannot run the model here).
classes = ['dress', 'hat', 'longsleeve', 'outwear', 'pants',
           'shirt', 'shoes', 'shorts', 'skirt', 't-shirt']
logits = np.array([-1.868, -4.761, -2.317, -1.063, 9.887,
                   -2.812, -3.666, 3.200, -2.602, -4.835])

# WHY: softmax turns any real-valued scores into positive numbers that sum to 1. It never changes the ranking,
#      so "pants wins" holds either way; it only matters when a client wants a confidence.
# PYTHON: subtracting the max before exp() avoids overflow for large logits; the result is identical.
exp = np.exp(logits - logits.max())
probs = exp / exp.sum()

pd.DataFrame({'logit': logits, 'probability': probs.round(4)}, index=classes).sort_values('logit', ascending=False)

## 10.3 Creating a pre-processing service

The notebook becomes `gateway.py` (`jupyter nbconvert --to script`), wrapped in Flask. Then the 1.7 GB TensorFlow dependency goes: the only thing the gateway used was `tf.make_tensor_proto`, so `proto.py` rebuilds it with the small `tensorflow-protobuf` package.

In [ ]:
show(COMMENTED / 'code/proto.py')

In [ ]:
# GOAL: confirm the NumPy 2 fix in proto.py, and see what tensor_content really is.
# WHY: tostring() was a deprecated alias of tobytes(), removed in NumPy 2.3 -> the lecture's proto.py crashes.
print('NumPy', np.__version__, '| ndarray.tostring exists:', hasattr(np.ndarray, 'tostring'))

# A fake 'image batch' with the model's input shape (1, 299, 299, 3).
X = np.random.default_rng(1).random((1, 299, 299, 3), dtype=np.float32)

# WHY: protobuf carries the raw bytes + shape + dtype. The server rebuilds the identical array from them.
# PYTHON: np.frombuffer reads bytes back as an array of the given dtype; reshape restores the dimensions.
raw = X.tobytes()
X_back = np.frombuffer(raw, dtype=np.float32).reshape(X.shape)

len(raw), len(raw) == 299 * 299 * 3 * 4, np.array_equal(X, X_back)   # 4 bytes per float32

In [ ]:
show(COMMENTED / 'code/gateway.py')

## 10.4 Running everything locally with Docker-compose

Two images: TF Serving with the model baked in, and the gateway. Run separately, the gateway fails with `UNAVAILABLE: failed to connect to all addresses`: inside its container `localhost` is the gateway itself. Compose puts both on one network where the **service name is the hostname**.

In [ ]:
show(COMMENTED / 'code/image-model.dockerfile')

In [ ]:
show(COMMENTED / 'code/image-gateway.dockerfile')

In [ ]:
show(COMMENTED / 'code/docker-compose.yaml')

```bash
# GOAL:   build both images with explicit tags (Kubernetes and compose refer to them by tag).
# SYNTAX: -t name:tag ; -f picks the Dockerfile ; '.' = build context.
docker build -t zoomcamp-10-model:xception-v4-001 -f image-model.dockerfile .
docker build -t zoomcamp-10-gateway:002 -f image-gateway.dockerfile .

# GOAL:   start / stop both together.
# SYNTAX: up -d = detached (background); down stops AND removes the containers and the network.
docker compose up -d
python test.py            # POSTs {"url": ...} to localhost:9696/predict
docker compose down
```

## 10.5 Introduction to Kubernetes

Kubernetes keeps a **desired state** (written in YAML) true: it starts, restarts and scales containers for you. The objects, with a factory-simulation reading:

| Object | What it is | Plant Simulation analogy |
|---|---|---|
| Node | a machine (an EC2 instance; with kind, a Docker container on your Mac) | the hall floor space a station can be placed on |
| Pod | one running container (sometimes a few tightly coupled ones) | one station instance doing the work |
| Deployment | N identical pods: same image, same config | a parametrised station object + its quantity |
| Service | stable name and port in front of the pods; spreads requests over them | the router / buffer in front of parallel stations |
| HPA | changes N with the measured load | capacity control that opens parallel stations when utilisation passes a target |
| Ingress | the cluster's front door, routes to external services | the plant gate |

Two service types matter here: **external** (`LoadBalancer`, gets a public address from the cloud) and **internal** (reachable only from pods in the cluster; the default when you omit `type`). Labels are the wiring: a deployment owns the pods carrying its label, and a service routes to the pods carrying its selector's label.

## 10.6 Deploying a simple service to Kubernetes

A Flask "ping" app, its image, a cluster with **kind** (Kubernetes **in** Docker: the whole cluster is one Docker container), then a deployment and a service. `kubectl` is the client that talks to any cluster, local or EKS.

In [ ]:
show(COMMENTED / 'code/ping/ping.py')
show(COMMENTED / 'code/ping/Dockerfile')

In [ ]:
show(COMMENTED / 'code/ping/deployment.yaml')

In [ ]:
show(COMMENTED / 'code/ping/service.yaml')

In [ ]:
# GOAL: check the wiring of a deployment + service pair the way Kubernetes will use it, BEFORE applying it.
# WHY: the classic silent failure: everything applies without error, but the service selector matches no pod
#      (typo in a label) or targetPort is not the port the container listens on -> 'connection refused'.
def wiring(deployment, service):
    # PYTHON: chained dict lookups walk down the YAML tree: spec -> template -> metadata -> labels.
    pod_labels = deployment['spec']['template']['metadata']['labels']
    selector = deployment['spec']['selector']['matchLabels']
    containers = deployment['spec']['template']['spec']['containers']
    container_ports = {p['containerPort'] for c in containers for p in c.get('ports', [])}

    svc_selector = service['spec']['selector']
    target_ports = [p.get('targetPort', p['port']) for p in service['spec']['ports']]

    # PYTHON: dict.items() <= other.items() is True when every key/value pair of the first is in the second.
    return {
        'deployment selector matches pod labels': selector.items() <= pod_labels.items(),
        'service selector matches pod labels': svc_selector.items() <= pod_labels.items(),
        'service targetPort is a containerPort': all(t in container_ports for t in target_ports),
        'service type': service['spec'].get('type', '(not set: default)'),
    }


ping_dep = load(LECTURE / 'ping/deployment.yaml')
ping_svc = load(LECTURE / 'ping/service.yaml')
wiring(ping_dep, ping_svc)

In [ ]:
# GOAL: prove the checker catches a mistake: break the label on a copy and run it again.
# PYTHON: copy.deepcopy copies nested dicts too; a plain dict.copy() would share the inner dicts.
import copy

broken = copy.deepcopy(ping_svc)
broken['spec']['selector'] = {'app': 'pong'}
wiring(ping_dep, broken)

The whole 10.6 sequence on your Mac. This practice cluster uses the default name `kind`; HW10 creates its own named cluster, so delete this one when you are done.

```bash
# GOAL:   build and test the image locally first. If it fails here, Kubernetes will not fix it.
# SYNTAX: build from the complete lecture folder (it has the Pipfile.lock).
cd notebooks/code/ping
docker build -t ping:v001 .
docker run -it --rm -p 9696:9696 ping:v001      # second terminal: curl localhost:9696/ping -> PONG

# GOAL:   a local cluster. kubectl's context becomes kind-kind (kind-<cluster name>).
kind create cluster
kubectl cluster-info --context kind-kind
kubectl get service                              # a fresh cluster already has one service

# GOAL:   deploy. Then load the image: the kind node cannot see Docker Desktop's images.
# WHY:    without `kind load` the pod shows ErrImagePull / ImagePullBackOff (kind tries Docker Hub).
kubectl apply -f deployment.yaml
kind load docker-image ping:v001
kubectl get pod                                  # wait for STATUS Running

# GOAL:   reach the pod, then the service. port-forward runs until Ctrl-C: use a second terminal for curl.
# SYNTAX: port-forward <target> LOCAL:REMOTE ; for a service, REMOTE is the SERVICE port (80), not 9696.
kubectl port-forward <pod-name> 9696:9696        # curl localhost:9696/ping
kubectl apply -f service.yaml
kubectl get service                              # ping: LoadBalancer, EXTERNAL-IP <pending> on kind
kubectl port-forward service/ping 8080:80        # curl localhost:8080/ping

# GOAL:   free the memory before HW10.
kind delete cluster
```

## 10.7 Deploying TensorFlow models to Kubernetes

The same pattern twice: a deployment + service for TF Serving (internal), a deployment + service for the gateway (external). The gateway finds the model through the service's **DNS name** `<service>.<namespace>.svc.cluster.local`. Remember: these repo files carry the EKS image addresses (see the comments).

In [ ]:
for name in ['model-deployment', 'model-service', 'gateway-deployment', 'gateway-service']:
    print('=' * 20, name)
    show(COMMENTED / f'code/kube-config/{name}.yaml')

In [ ]:
# GOAL: check both pairs, plus the link between them (gateway env var -> model service DNS name and port).
kc = LECTURE / 'kube-config'
model_dep, model_svc = load(kc / 'model-deployment.yaml'), load(kc / 'model-service.yaml')
gw_dep, gw_svc = load(kc / 'gateway-deployment.yaml'), load(kc / 'gateway-service.yaml')

# PYTHON: a dict comprehension builds {env var name: value} from the list of {name, value} entries.
env = {e['name']: e['value'] for e in gw_dep['spec']['template']['spec']['containers'][0]['env']}
host, port = env['TF_SERVING_HOST'].rsplit(':', 1)

expected_host = f"{model_svc['metadata']['name']}.default.svc.cluster.local"
model_ports = [p['port'] for p in model_svc['spec']['ports']]

pd.DataFrame({
    'model pair': wiring(model_dep, model_svc),
    'gateway pair': wiring(gw_dep, gw_svc),
}), host == expected_host, int(port) in model_ports

```bash
# GOAL:   the lesson's local run (with the LOCAL image tags in the deployment files, not the ECR ones).
kind load docker-image zoomcamp-10-model:xception-v4-001
kubectl apply -f model-deployment.yaml
kubectl apply -f model-service.yaml
kubectl port-forward service/tf-serving-clothing-model 8500:8500    # then: python gateway.py

kind load docker-image zoomcamp-10-gateway:002
kubectl apply -f gateway-deployment.yaml
kubectl apply -f gateway-service.yaml
kubectl port-forward service/gateway 8080:80                        # then test.py with port 8080

# GOAL:   debug from INSIDE the cluster: is the model service's DNS name reachable?
# WHY:    curl cannot talk gRPC; telnet only checks that something listens on the port.
kubectl exec -it <gateway-pod-name> -- bash
#   apt update && apt install -y telnet
#   telnet tf-serving-clothing-model.default.svc.cluster.local 8500
```

> Production caveat from the lesson: gRPC keeps long-lived connections, so Kubernetes' default service load balancing spreads gRPC traffic unevenly across model pods.

## Extra — the workshop flow (the shape of HW10)

The 2025 workshop replaces the two-tier TF Serving setup with **one** FastAPI service that runs an ONNX model in-process, and adds what a production deployment needs: resource **requests**, **liveness/readiness probes**, a `/health` endpoint and an **HPA**. HW10 is this flow with the HW5 lead-scoring image instead of the clothing classifier.

In [ ]:
show(COMMENTED / 'workshop/service/app.py')

In [ ]:
show(COMMENTED / 'workshop/service/Dockerfile')

In [ ]:
for name in ['deployment', 'service', 'hpa']:
    print('=' * 20, name)
    show(COMMENTED / f'workshop/k8s/{name}.yaml')

In [ ]:
# GOAL: put the lecture's ping deployment and the workshop's deployment side by side: what got added?
def summarise(deployment, service):
    c = deployment['spec']['template']['spec']['containers'][0]
    res = c.get('resources', {})
    # PYTHON: dict.get(key, default) avoids a KeyError when a field is absent; absent fields print as '-'.
    probe = lambda kind: c.get(kind, {}).get('httpGet', {}).get('path', '-')
    return {
        'image': c['image'],
        'imagePullPolicy': c.get('imagePullPolicy', '- (default)'),
        'replicas': deployment['spec'].get('replicas', '- (default 1)'),
        'containerPort': c['ports'][0]['containerPort'],
        'cpu request': res.get('requests', {}).get('cpu', '-'),
        'cpu limit': res.get('limits', {}).get('cpu', '-'),
        'readinessProbe': probe('readinessProbe'),
        'livenessProbe': probe('livenessProbe'),
        'service type': service['spec'].get('type', '- (default)'),
        'service port -> targetPort': ', '.join(f"{p['port']} -> {p.get('targetPort', p['port'])}"
                                                for p in service['spec']['ports']),
    }


ws_dep = load(WORKSHOP / 'k8s/deployment.yaml')
ws_svc = load(WORKSHOP / 'k8s/service.yaml')

pd.DataFrame({
    'lecture ping (10.6)': summarise(ping_dep, ping_svc),
    'workshop': summarise(ws_dep, ws_svc),
})

In [ ]:
wiring(ws_dep, ws_svc)

In [ ]:
# GOAL: the HPA's rule, so `kubectl get hpa -w` makes sense when the numbers move.
# WHY: utilisation is measured against the pods' cpu REQUEST (250m in the workshop): a pod using 375m runs at 150%.
#      No request -> no percentage -> TARGETS shows <unknown>. Same if metrics-server is missing.
def hpa_desired(current_replicas, current_util, target_util, min_replicas, max_replicas):
    # PYTHON: math.ceil rounds up: 2.1 pods needed -> 3 pods. min(max(...)) clamps into [min, max].
    desired = math.ceil(current_replicas * current_util / target_util)
    return min(max(desired, min_replicas), max_replicas)


hpa = load(WORKSHOP / 'k8s/hpa.yaml')['spec']
target = hpa['metrics'][0]['resource']['target']['averageUtilization']

# Toy load levels for 2 running pods (average CPU as % of request):
for util in [10, 50, 120, 400]:
    print(f"{util:>4}% -> {hpa_desired(2, util, target, hpa['minReplicas'], hpa['maxReplicas'])} replicas")

The workshop sequence (and, with other names, the HW10 sequence):

```bash
# GOAL:   build and smoke-test the image (fix the lockfile first: see the commented Dockerfile).
cd notebooks/workshop/service
uv lock
docker build -t clothing-classifier:v1 .
docker run -it --rm -p 8080:8080 clothing-classifier:v1         # curl localhost:8080/health

# GOAL:   a NAMED cluster: every kind command takes --name, every kubectl context is kind-<name>.
kind create cluster --name mlzoomcamp
kind load docker-image clothing-classifier:v1 --name mlzoomcamp

cd ../k8s
kubectl apply -f deployment.yaml
kubectl rollout status deployment/clothing-classifier           # waits for the readiness probe
kubectl apply -f service.yaml
kubectl port-forward service/clothing-classifier 30080:8080     # NodePort is not reachable on kind/macOS

# GOAL:   metrics-server, so the HPA can read CPU. kind's kubelet uses self-signed certificates,
#         hence the --kubelet-insecure-tls patch (local clusters only).
kubectl apply -f https://github.com/kubernetes-sigs/metrics-server/releases/latest/download/components.yaml
kubectl patch -n kube-system deployment metrics-server --type=json \
  -p '[{"op":"add","path":"/spec/template/spec/containers/0/args/-","value":"--kubelet-insecure-tls"}]'
kubectl get deployment metrics-server -n kube-system           # wait for 1/1
kubectl apply -f hpa.yaml
kubectl get hpa -w                                              # TARGETS should show a % after ~1 min
python ../load_test.py                                          # third terminal

# GOAL:   inspect and debug.
kubectl describe pod <pod-name>                                 # Events: image pull, probe failures, Pending reasons
kubectl logs -l app=clothing-classifier --tail=20
kubectl get events --sort-by='.lastTimestamp'

# GOAL:   clean up.
kind delete cluster --name mlzoomcamp
```

In [ ]:
show(COMMENTED / 'workshop/load_test.py')

## 10.8 Deploying to EKS (optional, costs money)

The same manifests work on a managed cluster. What changes: the cluster comes from `eksctl`, and images must be in a **registry** the nodes can pull from (ECR), because there is no `kind load` in the cloud. A `LoadBalancer` service now gets a real AWS load balancer with a public DNS name (open to anyone; restrict it in real projects).

In [ ]:
show(COMMENTED / 'code/kube-config/eks-config.yaml')

```bash
# GOAL:   create the cluster (15-20 min); eksctl also switches kubectl's context to it.
eksctl create cluster -f eks-config.yaml

# GOAL:   push the images to ECR. ECR tags cannot contain ':' after the repository, hence '-'.
aws ecr create-repository --repository-name mlzoomcamp-images
ACCOUNT_ID=<your account id>
REGION=eu-west-1
PREFIX=${ACCOUNT_ID}.dkr.ecr.${REGION}.amazonaws.com/mlzoomcamp-images
docker tag zoomcamp-10-gateway:002 ${PREFIX}:zoomcamp-10-gateway-002
# SYNTAX: get-login-password is the current login (the lesson's `aws ecr get-login` is AWS CLI v1 only).
aws ecr get-login-password --region ${REGION} | docker login --username AWS --password-stdin ${PREFIX%%/*}
docker push ${PREFIX}:zoomcamp-10-gateway-002

kubectl apply -f model-deployment.yaml -f model-service.yaml -f gateway-deployment.yaml -f gateway-service.yaml
kubectl get service                    # gateway EXTERNAL-IP = an ELB DNS name (needs a few minutes to resolve)

# GOAL:   stop paying. Same day, no exceptions. Also removes the load balancer.
eksctl delete cluster --name mlzoomcamp-eks
```

> Apple Silicon: images built on your Mac are `arm64`; `m5.xlarge` nodes are `amd64`. For EKS, build with `docker buildx build --platform linux/amd64 ...`, or the pods fail with `exec format error`. kind on your Mac is `arm64` too, so local images just work there.

## 10.9 Summary

| Piece | Does | Watch out |
|---|---|---|
| TF Serving | fast C++ inference for TF SavedModels over gRPC | needs `/models/<name>/<version>`; client needs protobuf |
| Gateway | pre/post-processing, JSON ↔ gRPC | keep heavy libraries out of it (`proto.py`) |
| docker-compose | several containers on one network, service name = hostname | `localhost` inside a container is that container |
| kind | a whole Kubernetes cluster inside one Docker container | `kind load docker-image` every new tag; `--name` on every kind command |
| Deployment | N identical pods; labels tie it to its pods | selector must match template labels |
| Service | stable name/port in front of pods | selector ↔ pod label, targetPort ↔ containerPort |
| Probes | readiness gates traffic, liveness restarts | the endpoint must exist (`/health`) |
| HPA | replicas follow CPU % of the **request** | needs metrics-server and a cpu request |
| port-forward | your Mac's port → a pod or service | service form uses the **service** port; blocks the terminal |
| EKS | managed cluster on AWS | registry instead of `kind load`; amd64 images; delete the same day |

## 10.10 Explore more

- Other local clusters: minikube, k3d, k3s, microk8s; Rancher Desktop; Docker Desktop's built-in Kubernetes. [Lens](https://k8slens.dev/) as a GUI.
- Namespaces (everything here ran in `default`).
- Deploy the Module 5 service with these manifests.
- The gRPC load-balancing article linked at the end of lesson 10.7.